# Creating a Simple Agent with Tracing

In [9]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [2]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [11]:
nutrition_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful assistant giving out nutrition advice.
    You give concise answers.
    """,
)

In [3]:
placement_trainer_agent = Agent(
    name="Placement Trainer Agent",
    instructions="""
    You are a supportive placement trainer who prepares students for job interviews.
    Start by asking about the student's target role, experience level, and interview type.
    Then run a realistic mock interview, asking one question at a time and waiting for
    the student's answer before continuing. Include behavioral, role-specific, and
    introductory questions as appropriate.

    After each answer, give concise, constructive feedback: identify what worked,
    suggest one or two specific improvements, and offer an example of a stronger
    response when useful. Encourage structured answers such as STAR for behavioral
    questions. Do not invent details about the student's experience; ask follow-up
    questions when information is missing. Adapt the difficulty to the student and
    finish with a brief summary of strengths and practice priorities.
    """,
)

Let's execute the Agent:

In [12]:
with trace("Simple Nutrition Agent"):
    result = await Runner.run(nutrition_agent, "How healthy are bananas?")

print(result)

RunResult:
- Last agent: Agent(name="Nutrition Assistant", ...)
- Final output (str):
    Overall: yes, bananas are healthy as part of a balanced diet.
    
    Key benefits
    - Rich in potassium (supports heart and blood pressure) and vitamin B6
    - Good source of vitamin C and dietary fiber
    - Quick, natural energy from carbs; portable and filling
    - Resistant starch when unripe (handy for gut health)
    
    Things to consider
    - Moderate sugar content; portion size matters if watching sugar or calories
    - For kidney disease or very high potassium needs, limit intake
    
    Typical serving
    - 1 medium banana (about 105 calories; ~27 g carbs; ~3 g fiber; good potassium and B6)
    
    Bottom line: a convenient, nutrient-dense snack or addition to meals.
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Streaming the answer to the screen, token by token

In [13]:
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are bananas?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Overall, bananas are healthy in moderation.

- Nutrients: good source of potassium, vitamin B6, vitamin C, and dietary fiber.
- Benefits: supports heart health, digestion, and energy; contains resistant starch when unripe.
- Considerations: ripe bananas have more sugars; portion size matters if watching sugar intake or blood sugar.
- cautions: people with kidney disease or on potassium-restricted diets should monitor intake; bananas are generally safe for most people and are fat-free.

A typical one medium banana fits well in a balanced diet.

_Good Job!_